# Lecture 08, Notebook 08: Analytic OLG DEQN, Persistent Simulation in Lux

The training cloud is rolled forward under the current Lux policy. The closed
form savings rates remain diagnostics only.

## Lecture 08, Notebook 08: Analytic 6-Agent OLG — DEQN (Persistent Simulation)

**Course:** Deep Learning for Solving and Estimating Dynamic Models in Economics and Finance
**Script reference:** §5.2 (the 6-agent analytic Krueger–Kübler OLG); §5.3 (DEQN mapping); §5.4 (KKT)
**Notebook role:** core
**Author:** Simon Scheidegger

*Julia/Lux preview of* `lectures/lecture_08_olg_models_deqns/code/lecture_08_08_OLG_Analytic_DEQN_persistent.ipynb`.

> **Run mode.** The checked-in run uses `RUN_MODE = "smoke"` with fixed `SEED = 0` for a fast structural check. Set `RUN_MODE` to `"teaching"` or `"production"` in the config cell below to reproduce slide-scale accuracy. (Every mode in this preview uses the same compact `(32, 16)` hidden-layer architecture; the full Python notebook switches to the 100/50 hidden-layer architecture of Appendix A.8 in production.)

> **Sampling mode.** This is the primary classroom variant: the training cloud is **generated by simulating** the economy under the current policy network. Each segment continues from the terminal states of the previous one, so the network is trained on its own ergodic distribution. The feedback-free ablation is `lecture_08_07_OLG_Analytic_DEQN_exogenous_Lux.ipynb`.

This notebook solves the six-age overlapping-generations model of Krueger and Kübler (2004): log utility, stochastic Cobb–Douglas production, four i.i.d. TFP/depreciation aggregate states, and a closed-form age-specific savings rate that the trained policy can be validated against. This is the primary classroom notebook for the analytic OLG.

The notebook mirrors the structure of the IRBC and benchmark-OLG notebooks: parameters, augmented state, neural-network policy transform, residual / loss construction, training-data generation, mini-batch training, out-of-sample residual tables, policy-stability checks, and validation against the closed-form savings rates.

In [ ]:
import Pkg
Pkg.activate(joinpath(@__DIR__, "..", "..", "..", "julia"))

using DLEFJulia
using Lux
using NNlib
using Optimisers
using Statistics

In [ ]:
RUN_MODE = "smoke"
SEED = 0
budgets = (
    smoke = (segments = 4, batch_size = 16, segment_length = 4, anchor_size = 16, eval_burn_in = 4, eval_length = 4),
    teaching = (segments = 80, batch_size = 128, segment_length = 8, anchor_size = 256, eval_burn_in = 32, eval_length = 64),
    production = (segments = 2_000, batch_size = 512, segment_length = 16, anchor_size = 2_048, eval_burn_in = 128, eval_length = 256),
)
hp = run_mode_budget(RUN_MODE; budgets)
rng = rng_from_seed(SEED)

## 1. Economic parameters

The Krueger–Kübler (2004) calibration: $A = 6$ overlapping agents, log utility, Cobb–Douglas production with capital share $\alpha = 0.3$, discount factor $\beta = 0.7$, only agent 1 supplies labor, and four i.i.d. aggregate states combining TFP $\eta \in \{0.95, 1.05\}$ and depreciation $\delta \in \{0.5, 0.9\}$ with equal probability ($\pi_{ss'} = 0.25$). Under these choices the model has a closed-form age-specific savings rate (see §8). Here `AnalyticOLGParams()` carries this calibration; `cloud = sample_analytic_olg_states(rng, params, hp.batch_size)` draws the **initial random feasible trajectory heads** from broad boxes that the simulation will roll forward.

### 2. State representation and augmented network inputs

The minimal state is the cohort capital vector together with the aggregate shock indicator ($1 + A = 7$ numbers for $A = 6$). To make learning easier we feed the network an **extended state** that pre-computes aggregates, per-agent income blocks, and the shock-transition probabilities:

$$
  \dim(\mathbf{s}_t) = \underbrace{1 + 4 + 2 + 5}_{12\text{ aggregate}} + \underbrace{4A}_{\text{per-agent}} + \underbrace{4}_{\boldsymbol{\pi}(z_t)} = 16 + 4A,
$$

which is $40$ for the analytic model. In Julia the feature width is `analytic_olg_feature_dim(params)` and the augmentation is built by `analytic_olg_features`; it is pure feature engineering — the equilibrium is unchanged.

### 3. Neural network and policy transformation

The policy network is an MLP from the 40-dimensional augmented state to the savings of cohorts $1,\ldots,A-1$,

$$
  \mathcal{N}_\theta(\mathbf{s}_t) \;\longrightarrow\; \bigl(\hat a_t^1, \ldots, \hat a_t^{A-1}\bigr), \qquad \hat a_t^h = k_{t+1}^{h+1}.
$$

A **sigmoid savings-fraction head** squashes the raw output to $\hat\beta_h \in (0,1)$ so savings $\hat a_t^h = \hat\beta_h \cdot \mathrm{inc}_t^h \ge 0$ and consumption $c_t^h = (1-\hat\beta_h)\,\mathrm{inc}_t^h \ge 0$ hold by construction; capital-market clearing then holds by construction too. In Lux the network is `make_mlp(analytic_olg_feature_dim(params), (32, 16), params.n_ages - 1; activation = NNlib.tanh)`, the policy head is inside `analytic_olg_policy_from_raw` / `analytic_olg_residual`, and `setup_training` initializes Float64 parameters with `Optimisers.Adam`.

> **In this preview.** The Python network (Appendix A.8) uses ReLU hidden units and zero-initializes the final `Dense` layer, so the initial raw output is `0` and the initial savings fraction is a neutral `sigmoid(0) = 0.5`. This Lux preview substitutes a smooth `NNlib.tanh` activation and keeps the default Lux (Glorot) initialization for the final layer, so the neutral 0.5-fraction start is not reproduced exactly.

In [ ]:
params = AnalyticOLGParams()
model = make_mlp(analytic_olg_feature_dim(params), (32, 16), params.n_ages - 1; activation = NNlib.tanh)
state = setup_training(rng_from_seed(SEED; offset = 1), model, Optimisers.Adam(0.002); parameter_type = Float64)
cloud = sample_analytic_olg_states(rng, params, hp.batch_size)

## 4. Residuals and loss

For each cohort $h = 1, \ldots, A-1$ the relative Euler residual is

$$
  \mathrm{EulerErr}_h(\mathbf{s}_t) \;=\;
  \frac{\beta\, \mathbb{E}_t\!\left[\, u'(c_{t+1}^{h+1})\, \bigl(R_{t+1} + 1 - \delta_{t+1}\bigr)\,\right]}
       {u'(c_t^h)} \;-\; 1,
$$

with $u(c) = \log c$, reported in relative form. The Euler-residual component of the loss is the mean over states of $\sum_h (\mathrm{EulerErr}_h)^2$, with the conditional expectation evaluated by direct summation over the four discrete shock states; no market-clearing term is needed (§3). In Julia this is `analytic_olg_residual`, wrapped by `analytic_loss` so the trainer sees `(loss, state)`.

> The full Python notebook adds two feasibility backstops to the trained loss — negative consumption and negative next-period aggregate capital — both scaled by `PENALTY_WEIGHT = 10.0`. This compact Julia helper keeps only the negative-consumption backstop at unit weight; the sigmoid savings-fraction head should keep these backstops inactive in the intended feasible region, but longer runs may be less aggressively pushed away from invalid states than in Python.

### Training-data construction, training loop, and policy-stability

In this persistent version the training states are **generated by simulating** the economy under the current network policy. Initial trajectory heads are random feasible states from broad boxes; after each segment the simulation continues from the terminal states of the previous segment.

The next cell defines the machinery and runs the segment loop:

- `draw_analytic_next_shocks` samples the next aggregate shock from the Markov transition row, and `simulate_analytic_segment` rolls the economy forward under the current policy via `analytic_olg_next_states`, returning the segment path and its terminal states.
- The loop feeds each simulated segment to `train_step!` (Adam, gradient clipping `max_grad_norm = 10`), then sets `cloud_head = cloud_end` — the explicit realization of the Python `X_segment, X_end = get_training_segment(X_start, model); X_start = X_end` continuation.
- **Policy-stability (§7).** The architecture has no calendar-time input, so a fixed weight vector defines a stationary recursive policy; the empirical question is whether SGD has stopped moving it. `analytic_policy_fingerprint` evaluates the policy on a fixed holdout `anchor_states`, and `relative_policy_drift_stats` reports `policy_drift_rms` / `policy_drift_max` after each segment.

> **In this preview.** Several optimization and robustness elements of the full Python notebook are compacted here. (i) Python runs `PASSES_PER_SEGMENT` passes of shuffled mini-batches of size `BATCH_SIZE` per segment; this preview takes a single full-batch gradient step on the whole segment per segment, so the `budgets` field named `batch_size` is the number of simulated trajectories (the cloud/track count), not an SGD mini-batch size. (ii) Python's simulator detects numerically invalid states (non-finite, capital out of range) and repairs them with fresh feasible-box draws; `simulate_analytic_segment` here has no such repair path (benign in the short smoke run). (iii) Python's §7 stability check reports a PASS/WARNING verdict once `policy_drift_rms` / `policy_drift_max` fall below `TIME_INVARIANCE_TOL_RMS` / `TIME_INVARIANCE_TOL_MAX`, and runs a fixed-shock policy-iteration diagnostic; this preview returns the raw drift statistics only and leaves the stability call to the reader.

In [ ]:
function draw_analytic_next_shocks(rng, states; params)
    z_current = clamp.(round.(Int, vec(states[1, :])), 1, length(params.tfp))
    z_next = similar(z_current)
    for (i, z) in pairs(z_current)
        u = rand(rng)
        cumulative = zero(eltype(params.transition))
        z_next[i] = length(params.tfp)
        for shock in eachindex(params.tfp)
            cumulative += params.transition[z, shock]
            if u <= cumulative
                z_next[i] = shock
                break
            end
        end
    end
    return z_next
end

function simulate_analytic_segment(rng, train_state, start_states, steps; params)
    n_tracks = size(start_states, 2)
    path = similar(start_states, size(start_states, 1), n_tracks * steps)
    current = copy(start_states)
    for t in 1:steps
        cols = ((t - 1) * n_tracks + 1):(t * n_tracks)
        path[:, cols] .= current
        z_next = draw_analytic_next_shocks(rng, current; params)
        current = analytic_olg_next_states(
            train_state.model, train_state.ps, train_state.st, current, z_next; params
        )
    end
    return path, current
end

function analytic_policy_fingerprint(train_state, states; params)
    raw, _ = train_state.model(analytic_olg_features(states; params), train_state.ps, train_state.st)
    policy = analytic_olg_policy_from_raw(raw, states; params)
    return vcat(log1p.(policy.savings), policy.fraction)
end

function relative_policy_drift_stats(previous, current)
    diff = current .- previous
    scale = 1 + sqrt(mean(abs2, previous))
    return (
        rms = sqrt(mean(abs2, diff)) / scale,
        max_abs = maximum(abs.(diff)) / scale,
    )
end

function analytic_shock_counts(states; params)
    z = clamp.(round.(Int, vec(states[1, :])), 1, length(params.tfp))
    return [count(==(shock), z) for shock in eachindex(params.tfp)]
end

analytic_loss(model, ps, st, batch) = begin
    pieces, st_new = analytic_olg_residual(model, ps, st, batch; params)
    return pieces.loss, st_new
end

train_result = let cloud_head = cloud,
        sim_rng = rng_from_seed(SEED; offset = 10_000),
        anchor_states = sample_analytic_olg_states(rng_from_seed(SEED; offset = 999_001), params, hp.anchor_size)
    initial_loss_local = loss_value(state, analytic_loss, cloud_head)
    history_local = NamedTuple[]
    anchor_previous = analytic_policy_fingerprint(state, anchor_states; params)
    last_segment_local = cloud_head
    for segment in 1:hp.segments
        segment_states, cloud_end = simulate_analytic_segment(sim_rng, state, cloud_head, hp.segment_length; params)
        metrics = train_step!(state, analytic_loss, segment_states; max_grad_norm = 10.0)
        monitor, _ = analytic_olg_residual(state.model, state.ps, state.st, segment_states; params)
        anchor_current = analytic_policy_fingerprint(state, anchor_states; params)
        drift = relative_policy_drift_stats(anchor_previous, anchor_current)
        anchor_previous = anchor_current
        learned_error = residual_summary(monitor.policy_error)
        append_metric!(
            history_local;
            segment,
            loss = metrics.loss,
            monitor_loss = monitor.loss,
            policy_drift_rms = drift.rms,
            policy_drift_max = drift.max_abs,
            mean_abs_policy_error = learned_error.mean_abs,
            max_abs_policy_error = learned_error.max_abs,
            shock_counts = analytic_shock_counts(segment_states; params),
        )
        cloud_head = cloud_end
        last_segment_local = segment_states
    end
    (initial_loss = initial_loss_local, history = history_local, cloud = cloud_head, last_segment = last_segment_local)
end
initial_loss = train_result.initial_loss
history = train_result.history
cloud_final = train_result.cloud
last_segment = train_result.last_segment

## 6. Final diagnostics and closed-form validation

The diagnostics report relative Euler residuals on two out-of-sample clouds — the **last training segment** and a **fresh simulated ergodic cloud** (`simulate_analytic_segment` with `eval_burn_in` states dropped) — for accuracy on the region induced by the learned policy. Mean and max absolute Euler residuals are dimensionless.

The trained policy is validated against the closed-form savings rates of Krueger and Kübler (2004),

$$
  \beta_h \;=\; \beta \cdot \frac{1 - \beta^{A - h}}{1 - \beta^{A - h + 1}},
  \qquad h = 1, \ldots, A - 1,
$$

with `rates = analytic_olg_closed_form_savings_rates(params)` and the learned deviation measured by `analytic_olg_policy_error`. The closed-form rates are a validation target, never a training target. (The full Python notebook also draws loss/residual plots and a savings-rate comparison, replaced here by the returned diagnostics; it additionally reports residuals on an **exogenous off-trajectory test cloud** for robustness away from the ergodic set, which this preview omits in favor of the two on-trajectory clouds above.)

In [ ]:
diagnostics, _ = analytic_olg_residual(state.model, state.ps, state.st, last_segment; params)
eval_start = sample_analytic_olg_states(rng_from_seed(SEED; offset = 777), params, hp.batch_size)
eval_path, _ = simulate_analytic_segment(
    rng_from_seed(SEED; offset = 888), state, eval_start, hp.eval_burn_in + hp.eval_length; params
)
eval_sim = eval_path[:, (hp.eval_burn_in * hp.batch_size + 1):end]
eval_diagnostics, _ = analytic_olg_residual(state.model, state.ps, state.st, eval_sim; params)
learned_error = residual_summary(diagnostics.policy_error)
eval_learned_error = residual_summary(eval_diagnostics.policy_error)
rates = analytic_olg_closed_form_savings_rates(params)

## Conclusion

This Lux preview trained the analytic 6-age OLG DEQN on its **own simulated ergodic set**: initial feasible heads, Markov shock draws, segment-continuation (`cloud_head → cloud_end`), Euler-residual loss, policy-drift monitoring, and validation against the closed-form savings rates $\beta_h$.

For a quick check keep `RUN_MODE = "smoke"`; use `"teaching"`/`"production"` for larger runs. The cell below returns a machine-checkable summary (closed-form rates, initial/final loss, Euler residuals on the last segment and the simulated eval cloud, learned-policy errors, and the final policy-drift statistics) for this run.

In [ ]:
(
    sampling = :persistent_simulation,
    transition = :markov_policy_simulation,
    states_per_segment = hp.batch_size * hp.segment_length,
    closed_form_savings_rates = rates,
    initial_loss = initial_loss,
    final_loss = history[end].monitor_loss,
    max_abs_euler = residual_summary(diagnostics.euler).max_abs,
    learned_policy_mean_abs_error = learned_error.mean_abs,
    learned_policy_max_abs_error = learned_error.max_abs,
    simulated_eval_loss = eval_diagnostics.loss,
    simulated_eval_policy_mean_abs_error = eval_learned_error.mean_abs,
    last_policy_drift_rms = history[end].policy_drift_rms,
    last_policy_drift_max = history[end].policy_drift_max,
    last_segment_shock_counts = history[end].shock_counts,
    simulated_eval_shock_counts = analytic_shock_counts(eval_sim; params),
    cloud_aggregate_capital_mean = sum(cloud_final[2:end, :]) / size(cloud_final, 2),
)